In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_data import prepare_prices


# 01 Stock Data

Start here. Choose the research settings. Each notebook reads the shared defaults from src/research_config.py and the explicit input filenames shown in its cells. The default files are the repository’s existing inputs; their historical universe bias remains explicitly recorded.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 2. Settings

Shared settings are defined in src/research_config.py. Insert input filenames below. After changing inputs or settings, restart the kernels and execute the modules in order.


In [ ]:
CONFIG = ResearchConfig().validate()
cfg = CONFIG
PRICE_FILE = "data/inputs/prices.parquet"
MEMBERSHIP_FILE = None
ALLOW_LEGACY_UNIVERSE = True
display(pd.Series(cfg.to_dict(), name="Research settings"))


## 3. Load and split prices

The first 70% of sessions form the training sample. Missingness is evaluated only there; test-period prices are never used to select assets. No live download occurs.


In [ ]:
prices = pd.read_parquet(PRICE_FILE)
membership = pd.read_csv(MEMBERSHIP_FILE) if MEMBERSHIP_FILE is not None else None
train_prices, test_prices, availability = prepare_prices(
    prices, cfg, membership, ALLOW_LEGACY_UNIVERSE
)
display(
    pd.DataFrame(
        {
            "observations": [len(train_prices), len(test_prices)],
            "start": [train_prices.index.min(), test_prices.index.min()],
            "end": [train_prices.index.max(), test_prices.index.max()],
        },
        index=["Formation", "Out of sample"],
    )
)
display(availability.head(10))


## 4. Inspect and save

The explicit filenames below are the inputs for subsequent modules. Existing files with those names are overwritten.


In [ ]:
train_prices.to_parquet("train_prices.parquet")
test_prices.to_parquet("test_prices.parquet")
availability.to_parquet("availability.parquet")
ax = (train_prices.iloc[:, :5] / train_prices.iloc[0, :5]).plot(
    figsize=(10, 4), title="Formation prices normalized to one"
)
ax.set_ylabel("Normalized price")
plt.show()
